# Module 8: Tests
### BWSI SGAI 2026

By the end of this module you'll know:
- How to **run the test suite** and read the output
- How to **read an existing test** to understand what it's verifying
- How to **add one assertion** to an existing test and confirm it passes

---
*Open this notebook in JupyterLab. Press `Alt+R` to enter slideshow mode.*  
*Students: just scroll through normally and run the exercise cells.*

## Why does this codebase have 293 tests?

Tests are the codebase's **safety net**.

When you change something — say, you edit the reward formula, or add a new action — you might accidentally break something unrelated. Tests catch that before it causes confusing game behavior. The cost is extra friction and potentially slower development. HINT: Claude can help with the boilerplate and machinery but you should make sure it's testing the right stuff in the right way. 

**Your relationship to tests in this module:**

| Goal | What you'll do |
|------|---------------|
| Know tests exist | Run `pytest tests/` — all 293 should pass |
| Understand what's tested | Read one test function, trace what it checks |
| Protect your own changes | Run tests before and after editing something |
| Add a small check | Append one assertion to an existing test |

You are **not** being asked to write test infrastructure from scratch. You're here to read, run, and make one small addition.

## Where tests live

```
bwsi-sgai/
├── gameplay/
├── models/
├── game_interfaces/
├── ui/
├── training/
└── tests/          <- this module
    ├── conftest.py
    ├── test_scorekeeper.py
    ├── test_humanoid.py
    ├── test_data_parser.py
    ├── test_data_model.py
    ├── test_cnn_classifier.py
    ├── test_oracle_classifier.py
    ├── test_heuristic_recommender.py
    ├── test_rl_recommender.py
    ├── test_dqn_agent.py
    ├── test_train_env.py
    ├── test_build_obs.py
    ├── test_integration.py
    ├── test_cli.py
    ├── test_ui_assets.py
    ├── test_ui_machine_advance.py
    ├── test_ui_pure_functions.py
    └── test_seed_reproducibility.py
```

*293 tests total. Every file is named `test_*.py` — pytest finds them automatically.*

*`bwsi-sgai/tests/`*

## What's in tests/ — Orient

The test suite is organized in three layers:

**Gameplay layer** (the rules)
| File | What it covers |
|------|---------------|
| `test_data_model.py` | Enum ordering and `ActionCost` values — pins the contracts the RL obs space depends on |
| `test_humanoid.py` | `Humanoid` construction, state predicates, converters |
| `test_scorekeeper.py` | Scoring math — save, squish, skip, scram, capacity, reward formula |
| `test_data_parser.py` | CSV loading, random draw, pool exhaustion, `datarow_to_value()` |

**Models + RL layer**
| File | What it covers |
|------|---------------|
| `test_cnn_classifier.py` | `classify()` output shape, probability range, softmax sum |
| `test_oracle_classifier.py` | Deterministic state-to-one-hot mapping |
| `test_heuristic_recommender.py` | All state × capacity combinations |
| `test_rl_recommender.py` | All 15 observation slots — range, encoding, time binning |
| `test_dqn_agent.py` | Construction, `select_action()`, `train_step()`, save/load round-trip |
| `test_train_env.py` | `TrainEnv.step()` — blocked actions, terminal step, obs shape/range |
| `test_build_obs.py` | `build_obs()` — verifies `TrainEnv` and `RLRecommender` produce identical observations |

**System + UI layer**
| File | What it covers |
|------|---------------|
| `test_integration.py` | End-to-end episode loop with each classifier/recommender combination |
| `test_cli.py` | `main.py` argument parsing — all modes, flags, invalid inputs |
| `test_ui_assets.py` | UI asset presence — logo and graphics files at expected paths |
| `test_ui_machine_advance.py` | Machine-mode advance button behavior |
| `test_ui_pure_functions.py` | Pure UI helper functions — no Tkinter window required |
| `test_seed_reproducibility.py` | Same seed produces identical humanoid sequences across runs |

*`bwsi-sgai/tests/`*

## What's in tests/ — Touch

Run the cell below to list the test files on your machine.

In [1]:
import os
import sys
from pathlib import Path
%matplotlib inline

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or lectures/.
_here = Path().resolve()
_root = _here.parent if _here.name == "lectures" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
import os

test_files = sorted(f for f in os.listdir('tests/') if f.startswith('test_'))
print(f"{len(test_files)} test files found:\n")
for f in test_files:
    print(f"  tests/{f}")

17 test files found:

  tests/test_build_obs.py
  tests/test_cli.py
  tests/test_cnn_classifier.py
  tests/test_data_model.py
  tests/test_data_parser.py
  tests/test_dqn_agent.py
  tests/test_heuristic_recommender.py
  tests/test_humanoid.py
  tests/test_integration.py
  tests/test_oracle_classifier.py
  tests/test_rl_recommender.py
  tests/test_scorekeeper.py
  tests/test_seed_reproducibility.py
  tests/test_train_env.py
  tests/test_ui_assets.py
  tests/test_ui_machine_advance.py
  tests/test_ui_pure_functions.py


*`bwsi-sgai/tests/conftest.py`*

## conftest.py — Shared Fixtures

Open `tests/conftest.py`. It's 224 lines.

**What's a fixture?**  
A fixture is a pre-built object that tests can request by name. Instead of every test creating its own `ScoreKeeper` or `Humanoid`, they ask for a fixture and get a fresh copy.

```python
# In conftest.py:
@pytest.fixture
def sk_fresh() -> ScoreKeeper:
    """Return a brand-new ScoreKeeper at full time with an empty ambulance."""
    return ScoreKeeper(shift_length=SHIFT_LENGTH, capacity=AMBULANCE_CAPACITY)

# In a test file:
def test_save_deducts_30_minutes(sk_fresh, h_healthy):
    before = sk_fresh.remaining_time
    sk_fresh.save(h_healthy)
    assert sk_fresh.remaining_time == before - 30
```

pytest sees `sk_fresh` in the function arguments and injects the fixture automatically.  
Each test gets its own isolated copy — one test can't mess up another's state.

*`bwsi-sgai/tests/conftest.py`*

## conftest.py — What's in it

```python
# ScoreKeeper fixtures
sk_fresh          # full time, empty ambulance
sk_half_time      # remaining_time == 360
sk_at_capacity    # ambulance completely full (10/10)
sk_near_end       # remaining_time == 5 (one squish left)

# Humanoid fixtures — one per state
h_zombie          # state="zombie", real image path
h_healthy         # state="healthy", real image path
h_injured         # state="injured", real image path
h_corpse          # state="corpse", real image path
humanoid_nonzero_value  # state="healthy", value=0.7 (for bin tests)

# Probabilities fixture
probs_healthy     # np.array([0.0, 1.0, 0.0, 0.0])  (one-hot healthy)

# DataParser fixture
data_parser       # DataParser(data_dir="data"), fresh pool per test

# Model fixtures — session-scoped (weights load once per test run)
cnn_classifier    # CNNClassifier with real baseline weights
rl_recommender    # RLRecommender with real DQN weights
```

`scope="session"` on the model fixtures means they load once for the whole test run — no repeated disk reads.

*`bwsi-sgai/tests/conftest.py`*

## conftest.py — Touch

Run the cell below to list all fixture names defined in `conftest.py`.

In [3]:
import ast
import os

with open('tests/conftest.py', encoding='utf-8') as f:
    source = f.read()

tree = ast.parse(source)
fixtures = []
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        for decorator in node.decorator_list:
            # Match @pytest.fixture and @pytest.fixture(scope=...)
            if isinstance(decorator, ast.Attribute) and decorator.attr == 'fixture':
                fixtures.append(node.name)
            elif isinstance(decorator, ast.Call):
                func = decorator.func
                if isinstance(func, ast.Attribute) and func.attr == 'fixture':
                    fixtures.append(node.name)

print(f"{len(fixtures)} fixtures in conftest.py:\n")
for name in fixtures:
    print(f"  {name}")

13 fixtures in conftest.py:

  sk_fresh
  sk_half_time
  sk_at_capacity
  sk_near_end
  h_zombie
  h_healthy
  h_injured
  h_corpse
  humanoid_nonzero_value
  probs_healthy
  data_parser
  cnn_classifier
  rl_recommender


## Now: reading a test function

```
bwsi-sgai/
├── gameplay/
├── models/
├── game_interfaces/
├── ui/
├── training/
└── tests/
    ├── conftest.py
    ├── test_scorekeeper.py   <- now
    └── ...
```

*`test_scorekeeper.py` covers `ScoreKeeper` — the most central class in the codebase.*  
*It's a good model for reading because the tests are short and the assertions are direct.*

*`bwsi-sgai/tests/test_scorekeeper.py`*

## Reading a Test — The Pattern

Every test in this codebase follows **Arrange / Act / Assert**:

```python
def test_save_deducts_30_minutes(sk_fresh, h_healthy):
    """Assert that save() deducts exactly 30 minutes from remaining_time."""
    # Arrange: fixtures sk_fresh and h_healthy are injected by pytest
    before = sk_fresh.remaining_time     # record the starting state

    # Act: call the thing being tested
    sk_fresh.save(h_healthy)

    # Assert: verify the outcome
    assert sk_fresh.remaining_time == before - 30
```

**What to notice:**
- Function name starts with `test_` — that's how pytest finds it
- Arguments are fixture names — pytest injects them automatically
- The docstring says exactly what the test proves
- One assertion per test is the norm — small, focused, easy to diagnose when it fails

*`bwsi-sgai/tests/test_scorekeeper.py`*

## Reading a Test — Touch

Run the cell below to print the source of one test function directly.

In [4]:
import inspect
import importlib.util

# Load the test module without pytest
spec = importlib.util.spec_from_file_location('test_scorekeeper', 'tests/test_scorekeeper.py')
mod = importlib.util.module_from_spec(spec)
# We can inspect it without executing (which needs fixtures)
with open('tests/test_scorekeeper.py', encoding='utf-8') as f:
    lines = f.readlines()

# Print one test function — locate it by scanning for the function def
target = 'def test_save_deducts_30_minutes'
start = None
for i, line in enumerate(lines):
    if target in line:
        start = i
        break

if start is not None:
    # Print until the next function or class def
    end = start + 1
    while end < len(lines) and not (lines[end].startswith('def ') or lines[end].startswith('class ')):
        end += 1
    print(''.join(lines[start:end]))

def test_save_deducts_30_minutes(sk_fresh, h_healthy):
    """Assert that save() deducts exactly 30 minutes from remaining_time."""
    before = sk_fresh.remaining_time
    sk_fresh.save(h_healthy)
    assert sk_fresh.remaining_time == before - 30





*`bwsi-sgai/tests/test_scorekeeper.py`*

## Test Names Tell You What They Check

Run the cell below to list all test function names in `test_scorekeeper.py`.  
Read them like a table of contents — each name is a claim about the codebase's behavior.

In [5]:
import ast

with open('tests/test_scorekeeper.py', encoding='utf-8') as f:
    source = f.read()

tree = ast.parse(source)
test_names = [
    node.name
    for node in ast.walk(tree)
    if isinstance(node, ast.FunctionDef) and node.name.startswith('test_')
]

print(f"{len(test_names)} tests in test_scorekeeper.py:\n")
for name in test_names:
    print(f"  {name}")

72 tests in test_scorekeeper.py:

  test_initial_state_is_clean
  test_save_deducts_30_minutes
  test_save_adds_to_ambulance
  test_save_zombie_adds_to_zombie_slot
  test_save_injured_adds_to_injured_slot
  test_squish_deducts_5_minutes
  test_squish_healthy_increments_killed
  test_squish_injured_increments_killed
  test_squish_zombie_does_not_increment_killed
  test_squish_corpse_does_not_increment_killed
  test_squish_increments_squished_counter
  test_skip_deducts_15_minutes
  test_skip_injured_increments_killed
  test_skip_healthy_does_not_increment_killed
  test_skip_zombie_does_not_increment_killed
  test_skip_corpse_does_not_increment_killed
  test_scram_deducts_120_minutes
  test_scram_with_no_zombie_increments_saved
  test_scram_with_zombie_counts_humans_as_killed
  test_scram_zeros_ambulance
  test_scram_corpses_silently_discarded
  test_at_capacity_true_when_full
  test_at_capacity_false_when_not_full
  test_available_action_space_save_disabled_at_capacity
  test_available_

> Each test name is a claim. If a test fails, the name tells you exactly which claim broke.  
> `test_squish_zombie_does_not_increment_killed` — you know without reading the body that squishing a zombie should not count as a kill.

## Running the Tests

Open a terminal in JupyterLab (File → New → Terminal) and run:

```bash
# Run the full suite
pytest tests/

# Run with coverage report
pytest tests/ --cov

# Run just one file
pytest tests/test_scorekeeper.py

# Run a specific test by name
pytest tests/test_scorekeeper.py::test_save_deducts_30_minutes
```

> Do **not** run `pytest` inside a notebook cell — it works, but the output is messy and it conflicts with Jupyter's own process management. Use the terminal.

The cell below shows what a passing run looks like.

In [6]:
# This is what a passing run looks like — NOT live pytest output.
# Run pytest in the terminal to get the real thing.

expected_output = """
========================= test session starts ==========================
platform win32 -- Python 3.13.x, pytest-9.0.3, pluggy-1.x
rootdir: C:\\Users\\...\\bwsi-sgai
collected 293 items

tests/test_build_obs.py ..........                              [  3%]
tests/test_cli.py ..............................                [ 13%]
tests/test_cnn_classifier.py .....                             [ 15%]
tests/test_data_model.py ..........                            [ 18%]
tests/test_data_parser.py ..............                       [ 23%]
tests/test_dqn_agent.py .............                          [ 28%]
tests/test_heuristic_recommender.py .................          [ 33%]
tests/test_humanoid.py ............                            [ 37%]
tests/test_integration.py ......                               [ 39%]
tests/test_oracle_classifier.py ......                         [ 41%]
tests/test_rl_recommender.py ..............................     [ 52%]
tests/test_scorekeeper.py ..................................................[ 68%]
tests/test_seed_reproducibility.py ......                      [ 70%]
tests/test_train_env.py ...................                     [ 76%]
tests/test_ui_assets.py ...                                    [ 77%]
tests/test_ui_machine_advance.py .....                         [ 79%]
tests/test_ui_pure_functions.py ..........................     [100%]

========================= 293 passed in 12.41s =========================
"""

print(expected_output)


========================= test session starts ==========================
platform win32 -- Python 3.13.x, pytest-9.0.3, pluggy-1.x
rootdir: C:\Users\...\bwsi-sgai
collected 293 items

tests/test_build_obs.py ..........                              [  3%]
tests/test_cli.py ..............................                [ 13%]
tests/test_cnn_classifier.py .....                             [ 15%]
tests/test_data_model.py ..........                            [ 18%]
tests/test_data_parser.py ..............                       [ 23%]
tests/test_dqn_agent.py .............                          [ 28%]
tests/test_heuristic_recommender.py .................          [ 33%]
tests/test_humanoid.py ............                            [ 37%]
tests/test_integration.py ......                               [ 39%]
tests/test_oracle_classifier.py ......                         [ 41%]
tests/test_rl_recommender.py ..............................     [ 52%]
tests/test_scorekeeper.py ................

## Reading Coverage Output

When you run `pytest tests/ --cov`, you get a coverage table. Here's how to read it:

```
Name                                    Stmts   Miss  Cover
-----------------------------------------------------------
gameplay/scorekeeper.py                   142     14    90%
gameplay/humanoid.py                       28      2    93%
models/CNNClassifier.py                    41      6    85%
models/DQN.py                              45      5    89%
main.py                                   112     61    45%
-----------------------------------------------------------
TOTAL                                    1024    192    81%
```

**What the numbers mean:**
- `Stmts` — total executable lines in the file
- `Miss` — lines that were never reached during the test run
- `Cover` — percentage of lines that ran at least once

**Why is `main.py` only 45%?**  
`main.py` launches Tkinter windows and game loops. Tests can't open a window without a display, so most of `main.py` is covered by CLI tests only — the game-launching paths are skipped. That's fine; 45% on code that's hard to test is expected.

> **81% overall** means 19% of lines were never executed during the test run.  
> That doesn't mean those lines are wrong — it means no test exercised that path.  
> If you add a new function and don't test it, coverage drops.

INSTRUCTOR NOTES:
Point out that 81% is respectable for a codebase of this size.
If students ask about the missing 19%: most of it is Tkinter UI code, error branches, and CLI edge cases.
Don't let students get distracted trying to hit 100% -- that's not the goal here.
The goal is: run tests before and after you change something.

## When to Run Tests: The Experimental Workflow

For your research project, tests are a **sanity check**, not a grade.

**Before making any change:**
```bash
pytest tests/
# All 293 pass? Good. That's your baseline.
```

**After making a change:**
```bash
pytest tests/
# Any new failures? That tells you what broke.
```

**When debugging a failure, run just the relevant file:**
```bash
pytest tests/test_scorekeeper.py -v
# -v gives verbose output — each test name on its own line
```

This pattern is called **regression testing** — you're checking that new changes didn't break old behavior. It catches *silent* regressions: bugs that don't crash anything but change the game's behavior in a way you didn't intend.

## What a Regression Looks Like

Suppose you're editing `ScoreKeeper.squish()` to add logging, and you accidentally change the time deduction from 5 to 15. The game still runs. It doesn't crash.

But `pytest` will catch it:

```
FAILED tests/test_scorekeeper.py::test_squish_deducts_5_minutes

AssertionError:
  assert 705 == 715
         ^^^   ^^^
  remaining_time  expected
```

That failure tells you exactly: `squish()` should have taken 5 minutes, but it took 15.  
Without the test, you'd only notice this during a game if you happened to run out of time early — and even then you might not know why.

**Tests make invisible bugs visible.**

## Exercise: Run and Read

```
bwsi-sgai/
├── gameplay/
├── models/
├── game_interfaces/
├── ui/
├── training/
└── tests/
    ├── conftest.py
    └── test_scorekeeper.py   <- exercise target
```

*Three parts. Total time: under 10 minutes.*

## Exercise Part 1 — Run the Suite

Open a terminal in JupyterLab (File → New → Terminal) and run:

```bash
pytest tests/ --cov
```

**What to look at:**
1. The last line: `293 passed in X.XXs` — all green means your environment is clean
2. The coverage table: find `main.py` — it should say around `45%`
3. Find the file with the highest coverage (probably `gameplay/humanoid.py` or `gameplay/enums.py`)

If any test **fails**: stop and let your instructor know before continuing. A test failure at baseline means something in your environment differs from the reference setup.

## Exercise Part 2 — Add One Assertion

Open `tests/test_scorekeeper.py` in JupyterLab (left sidebar → navigate to `tests/` → double-click).

Find `test_initial_state_is_clean`. It looks like this:

```python
def test_initial_state_is_clean(sk_fresh):
    """Assert that a fresh ScoreKeeper has an empty ambulance, zero score, and full remaining time."""
    assert sk_fresh.remaining_time == SHIFT_LENGTH
    assert sk_fresh.score["killed"] == 0
    assert sk_fresh.score["saved"] == 0
    assert sum(sk_fresh.ambulance.values()) == 0
```

**Add one more assertion at the end:**

```python
    assert sk_fresh.remaining_time >= 0   # time is never negative at start
```

Save the file. Then run:

```bash
pytest tests/test_scorekeeper.py::test_initial_state_is_clean -v
```

You should see: `PASSED`.

INSTRUCTOR NOTES:
Verify students can open the file in the JupyterLab editor.
Some students may have trouble finding the test file -- remind them to use the left sidebar.
The assertion they're adding is trivially true -- that's intentional. The goal is the mechanical act of
adding an assertion and confirming it passes, not writing a clever test.
If students finish early, have them try adding an assertion that FAILS (e.g., assert sk_fresh.remaining_time == 999)
so they see what a test failure output looks like.

## Exercise Part 3 — Challenge: Find Your Research Test

Think about the function you're most likely to modify in your research project. For example:

- Changing the reward formula → `gameplay/scorekeeper.py` → tests in `test_scorekeeper.py`
- Changing `datarow_to_value()` → `game_interfaces/data_parser.py` → tests in `test_data_parser.py`
- Editing a heuristic rule → `models/HeuristicRecommender.py` → tests in `test_heuristic_recommender.py`

**Your task:**

1. Open the corresponding test file
2. Find one test that covers the function you plan to modify
3. Read it. Answer these questions:
   - What does the test assert?
   - If you change the function's behavior, will this test still pass?
   - What assertion would you need to add to cover your new behavior?

You don't have to write the new assertion now. Just identify the test and read it.

## Exercise Part 3 — Scratch Space

Use the cell below to explore the test file for your target function.

In [7]:
# List all test names in the test file for your target
# Change 'test_scorekeeper' to the file you care about

import ast

target_file = 'tests/test_scorekeeper.py'   # <- change this

with open(target_file, encoding='utf-8') as f:
    source = f.read()

tree = ast.parse(source)
test_names = [
    node.name
    for node in ast.walk(tree)
    if isinstance(node, ast.FunctionDef) and node.name.startswith('test_')
]

print(f"{len(test_names)} tests in {target_file}:\n")
for name in test_names:
    print(f"  {name}")

72 tests in tests/test_scorekeeper.py:

  test_initial_state_is_clean
  test_save_deducts_30_minutes
  test_save_adds_to_ambulance
  test_save_zombie_adds_to_zombie_slot
  test_save_injured_adds_to_injured_slot
  test_squish_deducts_5_minutes
  test_squish_healthy_increments_killed
  test_squish_injured_increments_killed
  test_squish_zombie_does_not_increment_killed
  test_squish_corpse_does_not_increment_killed
  test_squish_increments_squished_counter
  test_skip_deducts_15_minutes
  test_skip_injured_increments_killed
  test_skip_healthy_does_not_increment_killed
  test_skip_zombie_does_not_increment_killed
  test_skip_corpse_does_not_increment_killed
  test_scram_deducts_120_minutes
  test_scram_with_no_zombie_increments_saved
  test_scram_with_zombie_counts_humans_as_killed
  test_scram_zeros_ambulance
  test_scram_corpses_silently_discarded
  test_at_capacity_true_when_full
  test_at_capacity_false_when_not_full
  test_available_action_space_save_disabled_at_capacity
  test_avai

## Pop Quiz — Navigation

These are navigation questions. Which file do you open?

---

**Q1.** You changed `datarow_to_value()` in `game_interfaces/data_parser.py` to return values in `[0, 2]` instead of `[0, 1]`. Which test file would you check to see if a test already covers this?

A) `tests/test_scorekeeper.py`  
B) `tests/test_data_parser.py`  
C) `tests/test_build_obs.py`  
D) `tests/test_train_env.py`

**Answer: B — `tests/test_data_parser.py`**  
`datarow_to_value()` lives in `data_parser.py`, so its tests live in `test_data_parser.py`. The test file name always mirrors the source file name.

## Pop Quiz — Navigation

**Q2.** Your team wants a shared `DataParser` object that multiple new tests can use without each test rebuilding it. Where do you add it?

A) At the top of each test file, as a module-level variable  
B) `tests/conftest.py` — as a new `@pytest.fixture`  
C) `gameplay/constants.py` — as a shared configuration object  
D) Inside each test function that needs it

**Answer: B — `tests/conftest.py`**  
Shared test objects belong in `conftest.py` as fixtures. pytest injects them automatically; each test gets a fresh instance. One already exists: `data_parser` — you can use it immediately without adding anything.

## Pop Quiz — Navigation

**Q3.** `pytest tests/` reports `293 passed`. Does that mean the game is correct?

A) Yes — all tests pass means all behavior is verified  
B) Mostly — 81% coverage means only a small amount of code is untested  
C) Not necessarily — tests only verify what was tested; new behaviors may have no test yet  
D) No — the tests are probably wrong too

**Answer: C — Not necessarily**  
293 tests pass, but 81% coverage means 19% of code paths were never executed during the test run. More importantly: tests only check the behaviors that were *anticipated* when the tests were written. If you add a new behavior and don't add a test for it, 293 tests will still pass — and your new behavior is completely unchecked.

## What You Now Know

- `tests/` — 293 tests across 17 files, organized by the layer they test (gameplay, models, system)
- `tests/conftest.py` — shared fixtures; request them by name in any test function argument
- `pytest tests/` — run the full suite; `--cov` adds a coverage report
- `pytest tests/test_<file>.py::test_<name>` — run one specific test
- Arrange / Act / Assert — the pattern every test in this codebase follows
- Run tests **before and after** every change — any new failure tells you what broke

---

**Next: Module 9 — AI-Assisted Exploration (Claude Code)**  
How to use Claude to explain a function, suggest an extension, and debug a result — anchored to this specific codebase.